<a href="https://colab.research.google.com/github/UTD2026/Mixed_Dataset_Testing_STA/blob/main/Dynamic_V2_CE_Routing_With_Compute_ZipPathFix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V2-scaled dynamic CE routing + compute timing

This notebook runs the full A/B pipeline from a clean clone of `https://github.com/UTD2026/rishabh-tlm.git` when available, or from the uploaded repo ZIP fallback when cloning fails, then replaces the fixed CE-routing cutoff with a **V2-frac-scaled dynamic CE threshold**.

It now also records compute cost so the final table shows both quality and price:

1. Build the V2 admission pool and ledger with `cuda_ttl/ab_routing/ab_build_pool.py`.
2. Train the TTL LoRA adapter with `ab_train_ttl.py`.
3. Generate base and adapted predictions with `ab_generate.py`.
4. Evaluate fixed CE routing with `ab_route_eval.py`.
5. Sweep V2 multipliers with `ab_dynamic_ce_from_v2.py`.
6. Report accuracy, routed fraction, implied CE threshold, measured wall-clock stage time, and estimated routed generation cost.

Important leakage note: choosing the best multiplier on the same split is an analysis upper bound. For a clean paper result, tune the multiplier on validation, freeze it, and report test.

## Codebase walk-through

Relevant files in the repo snapshot:

- `cuda_ttl/ab_routing/ab_build_pool.py` is the key bridge. It reproduces Ruiyu's AC-GTP-LoRA V2 streaming admission selection and ledger, while also writing `features.jsonl`. That file contains `route_choice`, `q_ce`, `out_ce`, `admission_score`, and `in_final_pool` for every item.
- `cuda_ttl/ab_routing/ruiyu_ledger.py` defines V2 routing semantics: rows admitted when seen route to adapted; skipped/rejected rows route to base; init rows are resolved by final pool membership.
- `cuda_ttl/ab_routing/ab_train_ttl.py` trains the TTL LoRA on `selection.json` from the V2 pool and saves a merged model for vLLM.
- `cuda_ttl/ab_routing/ab_generate.py` produces base and adapted greedy predictions.
- `cuda_ttl/ab_routing/ab_route_eval.py` currently evaluates static arms such as `v2_admission` and fixed `ce_out@0.3`, `ce_out@0.5`, etc.
- `mlx_ttl/train_ttl_mlx.py` still has the older default training CE gate `--threshold 3.0`; that is separate from the inference-time CE routing threshold used here.

This notebook adds a reusable dynamic evaluator, rather than editing the original files destructively.

In [ ]:
# ---- user knobs ----
REPO_URL = "https://github.com/UTD2026/rishabh-tlm.git"
WORK = "/content/v2_dynamic_ce"  # change if not on Colab

# Clone fallback knobs. Useful if the GitHub repo is private, temporarily unavailable,
# or Colab has no GitHub auth configured. If git clone fails, the setup cell will
# look for this zip, then any rishabh-tlm*.zip in WORK / /content / current cwd.
# In Colab, upload the zip when prompted if none is found.
LOCAL_REPO_ZIP = "/content/rishabh-tlm-ab-routing-2026-07-02(1).zip"
GITHUB_TOKEN_ENV = "GITHUB_TOKEN"  # optional: set os.environ["GITHUB_TOKEN"] for private repos
MODEL = "Qwen/Qwen3.5-0.8B"
DATASETS = ["logiqa", "medicine", "geography"]
N_BY_DATASET = {"logiqa": 1000, "medicine": 1000, "geography": 225}

# Dynamic sweep. The example table came from 0.25x steps.
MULTIPLIERS = [round(x * 0.25, 2) for x in range(1, 25)]  # 0.25x ... 6.00x
FIXED_CE_TAU = 0.3

# Set True for a clean full run. It is GPU-heavy because it trains/generates.
# If False, the notebook can still replay against existing routing_eval JSONs if present.
RUN_FULL_PIPELINE = True
SKIP_IF_EXISTS = True

print({
    "repo": REPO_URL,
    "work": WORK,
    "model": MODEL,
    "datasets": DATASETS,
    "n_by_dataset": N_BY_DATASET,
    "run_full_pipeline": RUN_FULL_PIPELINE,
})

## Environment setup

This is written for a CUDA GPU box. On Colab/H200/Ada, the install below is enough for the repo's CUDA A/B scripts. If your image already has these packages, you can skip the install cell.

In [ ]:
import os, sys, subprocess, json, textwrap, shutil, pathlib, time
from pathlib import Path

Path(WORK).mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
print("cwd:", os.getcwd())

In [ ]:
# Optional install. Uncomment in a fresh runtime.
# !python -m pip install -U pip
# !python -m pip install "vllm" "peft" "transformers" "accelerate" "scipy" "sympy" "ninja" "pandas" "matplotlib"

In [ ]:
# Clone / update repo, with token + zip fallback
import os, subprocess, shutil, zipfile, glob
from pathlib import Path

repo_dir = Path(WORK) / "rishabh-tlm"


def _run_capture(cmd, cwd=None):
    """Run a shell command and return CompletedProcess while preserving stdout/stderr."""
    return subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )


def _clone_url_with_optional_token(url: str) -> str:
    token = os.environ.get(GITHUB_TOKEN_ENV, "").strip()
    if token and url.startswith("https://github.com/"):
        # Do not print this URL; it contains a secret.
        return url.replace("https://", f"https://{token}@", 1)
    return url


def _unzip_repo(zip_path: Path, dest: Path):
    zip_path = Path(zip_path).expanduser().resolve()
    if not zip_path.exists():
        raise FileNotFoundError(f"Repo zip does not exist: {zip_path}")

    print(f"Using repo zip fallback: {zip_path}")
    tmp = dest.parent / "_repo_zip_extract_tmp"
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(tmp)

    # Common cases: zip contains a single top-level folder, or contains repo files directly.
    candidates = [p for p in tmp.iterdir() if p.is_dir()]
    selected = None
    for cand in candidates:
        if (cand / "cuda_ttl" / "ab_routing").exists() or (cand / ".git").exists():
            selected = cand
            break
    if selected is None:
        if (tmp / "cuda_ttl" / "ab_routing").exists():
            selected = tmp
        elif len(candidates) == 1:
            selected = candidates[0]
        else:
            raise RuntimeError(
                "Could not identify repo root inside zip. "
                f"Top-level entries: {[p.name for p in tmp.iterdir()][:20]}"
            )

    if dest.exists():
        shutil.rmtree(dest)
    if selected == tmp:
        shutil.copytree(tmp, dest, dirs_exist_ok=True)
        shutil.rmtree(tmp, ignore_errors=True)
    else:
        shutil.move(str(selected), str(dest))
        shutil.rmtree(tmp, ignore_errors=True)

    if not (dest / "cuda_ttl" / "ab_routing").exists():
        raise RuntimeError(f"Unzipped repo does not contain cuda_ttl/ab_routing: {dest}")


def _find_repo_zip():
    """Find an already-present repo zip in the locations Colab commonly uses."""
    explicit = Path(LOCAL_REPO_ZIP).expanduser()
    search_roots = [
        explicit.parent,
        Path(WORK),
        Path.cwd(),
        Path("/content"),
        Path("/mnt/data"),  # works in ChatGPT sandbox, harmless in Colab if absent
    ]

    candidates = []
    if explicit.exists():
        candidates.append(explicit)
    for root in search_roots:
        try:
            if root.exists():
                candidates.extend(Path(x) for x in glob.glob(str(root / "rishabh-tlm*.zip")))
        except Exception:
            pass

    # de-dupe while preserving order
    out = []
    seen = set()
    for c in candidates:
        try:
            resolved = c.resolve()
        except Exception:
            continue
        if c.exists() and resolved not in seen:
            out.append(c)
            seen.add(resolved)
    return out[0] if out else None


def _upload_repo_zip_colab():
    """Upload a zip in Colab and return the actual path it was written to.

    Colab saves uploads into the current working directory, not necessarily /content.
    The previous notebook assumed /content/<name>, which caused FileNotFoundError.
    This version verifies the path and falls back to writing uploaded bytes manually.
    """
    from google.colab import files

    print("No local repo zip found. Upload rishabh-tlm-ab-routing-2026-07-02.zip now.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_names:
        raise RuntimeError("No .zip file uploaded.")

    name = zip_names[0]
    candidates = [
        Path.cwd() / name,
        Path(WORK) / name,
        Path("/content") / name,
        Path(name),
    ]
    for candidate in candidates:
        if candidate.exists():
            print(f"Uploaded zip found at: {candidate.resolve()}")
            return candidate.resolve()

    # Defensive fallback: Colab returned bytes, so write them somewhere we control.
    fallback = Path(WORK) / name
    fallback.parent.mkdir(parents=True, exist_ok=True)
    fallback.write_bytes(uploaded[name])
    print(f"Uploaded zip bytes written to: {fallback.resolve()}")
    return fallback.resolve()


if repo_dir.exists():
    print("repo already exists:", repo_dir)
    # keep this non-destructive; uncomment if you explicitly want latest remote
    # subprocess.check_call(["git", "-C", str(repo_dir), "pull"])
else:
    clone_url = _clone_url_with_optional_token(REPO_URL)
    print("Cloning repo...")
    cp = _run_capture(["git", "clone", clone_url, str(repo_dir)])
    if cp.returncode != 0:
        print("git clone failed.")
        if cp.stdout.strip():
            print("--- git stdout ---")
            print(cp.stdout[-4000:])
        if cp.stderr.strip():
            print("--- git stderr ---")
            # Redact token if present.
            safe_stderr = cp.stderr
            token = os.environ.get(GITHUB_TOKEN_ENV, "").strip()
            if token:
                safe_stderr = safe_stderr.replace(token, "<GITHUB_TOKEN>")
            print(safe_stderr[-4000:])

        zip_path = _find_repo_zip()
        if zip_path is None:
            try:
                zip_path = _upload_repo_zip_colab()
            except Exception as upload_e:
                raise RuntimeError(
                    "Could not clone the repo and no repo zip fallback was available.\n\n"
                    "Most likely causes:\n"
                    "1) The GitHub repo is private and Colab has no auth. Set os.environ['GITHUB_TOKEN'] first.\n"
                    "2) The URL/branch is wrong or unavailable.\n"
                    "3) Colab/network blocked GitHub.\n\n"
                    f"Original git error:\n{cp.stderr[-2000:]}"
                ) from upload_e

        _unzip_repo(Path(zip_path), repo_dir)

os.chdir(repo_dir)
print("repo:", repo_dir)
try:
    head = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, stderr=subprocess.DEVNULL).strip()
    print("git head:", head)
except Exception:
    print("git head unavailable: repo loaded from zip or non-git folder")

print("ab_routing files:")
for p in sorted((repo_dir / "cuda_ttl" / "ab_routing").glob("ab_*.py")):
    print(" -", p.relative_to(repo_dir))


In [ ]:
# Data paths expected by cuda_ttl/ab_routing/run_ab.sh, but we call scripts directly.
DATA_FILES = {
    "logiqa": "data/AdaptEval/logiqa_random_5k.json",
    "medicine": "data/AdaptEval/medicine_mcqa_random_5k.json",
    "geography": "data/AdaptEval/geography_mmlu.json",
    "gsm8k": "data/AdaptEval/gsm8k_random_5k.json",
}

missing = [p for ds, p in DATA_FILES.items() if ds in DATASETS and not Path(p).exists()]
if missing:
    raise FileNotFoundError(
        "Missing dataset files: " + ", ".join(missing) +
        "\nThe uploaded repo snapshot included these under data/AdaptEval/. If your clone does not, copy them there before running."
    )

for ds in DATASETS:
    print(ds, "->", DATA_FILES[ds], "exists", Path(DATA_FILES[ds]).exists())

## Add dynamic V2-scaled CE evaluator

This script consumes the same artifacts as `ab_route_eval.py` but adds the multiplier sweep:

- `v2_frac`: fraction routed to adapted by V2 admission ledger.
- `multiplier`: how much less conservative than V2 we allow CE-routing to be.
- `adapted_frac`: `v2_frac * multiplier`.
- `implied_tau`: output-CE cutoff induced by that adapted fraction.
- `peak_acc`: best routed accuracy on that multiplier grid.

Important: choosing the best multiplier from the test labels is an analysis upper bound. For a real paper table, pick `m` on a validation split and report the held-out test result.

In [ ]:
from pathlib import Path
script_path = Path("cuda_ttl/ab_routing/ab_dynamic_ce_from_v2.py")
script_path.write_text('#!/usr/bin/env python\n"""V2-frac-scaled dynamic CE routing evaluator.\n\nThis is a non-destructive extension of cuda_ttl/ab_routing/ab_route_eval.py.\nIt keeps CE-routing as the actual routing signal, but uses V2\'s admission-route\nfraction as the dynamic anchor for the CE threshold.\n"""\nimport argparse\nimport json\nimport math\nimport os\nimport sys\nimport time\nfrom pathlib import Path\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport grading  # noqa: E402\n\n\ndef load_jsonl(path):\n    with open(path, encoding="utf-8") as f:\n        return [json.loads(line) for line in f if line.strip()]\n\n\ndef graded(path, answer_type):\n    out = {}\n    raw = {}\n    for row in load_jsonl(path):\n        ex = grading.extract(str(row.get("predict", "")), answer_type)\n        out[int(row["idx"])] = bool(grading.is_correct(ex.value, str(row["label"]), answer_type))\n        raw[int(row["idx"])] = row\n    return out, raw\n\n\ndef route_acc(idx, base_ok, adapted_ok, to_adapted):\n    return sum((adapted_ok[i] if i in to_adapted else base_ok[i]) for i in idx) / max(1, len(idx))\n\n\ndef top_ce_set(idx, out_ce, adapted_frac):\n    """Return top adapted_frac of idx by output CE, plus implied tau."""\n    n = len(idx)\n    k = int(round(n * max(0.0, min(1.0, adapted_frac))))\n    order = sorted(idx, key=lambda i: out_ce[i], reverse=True)\n    chosen = set(order[:k])\n    if k <= 0:\n        tau = math.inf\n    elif k >= n:\n        tau = min(out_ce[i] for i in idx)\n    else:\n        tau = out_ce[order[k - 1]]\n    return chosen, tau, k / max(1, n)\n\n\ndef compute(dataset, base_preds, adapted_preds, features, multipliers, fixed_tau=0.3, routed_out=None):\n    t0 = time.perf_counter()\n    answer_type = grading.detect_answer_type(dataset)\n    if answer_type is None:\n        raise SystemExit(f"{dataset} is not a deterministic/verifiable bench known to grading.py")\n\n    base_ok, base_rows = graded(base_preds, answer_type)\n    adapted_ok, adapted_rows = graded(adapted_preds, answer_type)\n    feats = {int(r["idx"]): r for r in load_jsonl(features)}\n    idx = sorted(i for i in base_ok if i in adapted_ok and i in feats and feats[i].get("out_ce") is not None)\n    if not idx:\n        raise SystemExit("No overlapping rows across base/adapted/features")\n\n    out_ce = {i: float(feats[i]["out_ce"]) for i in idx}\n    v2_set = {i for i in idx if feats[i].get("route_choice") == "adapted"}\n    v2_frac = len(v2_set) / len(idx)\n\n    fixed_set = {i for i in idx if out_ce[i] >= fixed_tau}\n    fixed_acc = route_acc(idx, base_ok, adapted_ok, fixed_set)\n    v2_acc = route_acc(idx, base_ok, adapted_ok, v2_set)\n    base_acc = sum(base_ok[i] for i in idx) / len(idx)\n    adapted_acc = sum(adapted_ok[i] for i in idx) / len(idx)\n    oracle_acc = sum((base_ok[i] or adapted_ok[i]) for i in idx) / len(idx)\n\n    rows = []\n    for m in multipliers:\n        target_frac = min(1.0, v2_frac * float(m))\n        chosen, tau, actual_frac = top_ce_set(idx, out_ce, target_frac)\n        rows.append({\n            "dataset": dataset,\n            "multiplier": float(m),\n            "v2_frac": v2_frac,\n            "adapted_frac_target": target_frac,\n            "adapted_frac": actual_frac,\n            "implied_tau": tau,\n            "acc": route_acc(idx, base_ok, adapted_ok, chosen),\n            "n": len(idx),\n            "n_adapted": len(chosen),\n        })\n\n    # Tie-breaker: choose smaller adapted fraction if accuracy ties, then smaller multiplier.\n    best = sorted(rows, key=lambda r: (-r["acc"], r["adapted_frac"], r["multiplier"]))[0]\n\n    summary = {\n        "dataset": dataset,\n        "n": len(idx),\n        "answer_type": answer_type,\n        "v2_frac": v2_frac,\n        "v2_acc": v2_acc,\n        "all_base_acc": base_acc,\n        "all_adapted_acc": adapted_acc,\n        "oracle_acc": oracle_acc,\n        "fixed_tau": fixed_tau,\n        "fixed_ce_acc": fixed_acc,\n        "fixed_ce_adapted_frac": len(fixed_set) / len(idx),\n        "best_multiplier": best["multiplier"],\n        "best_adapted_frac": best["adapted_frac"],\n        "best_implied_tau": best["implied_tau"],\n        "peak_acc": best["acc"],\n        "dynamic_eval_seconds": None,\n        "sweep": rows,\n    }\n\n    if routed_out:\n        chosen, tau, _ = top_ce_set(idx, out_ce, best["adapted_frac"])\n        Path(routed_out).parent.mkdir(parents=True, exist_ok=True)\n        with open(routed_out, "w", encoding="utf-8") as f:\n            for i in idx:\n                row = dict(adapted_rows[i] if i in chosen else base_rows[i])\n                row["routed_to"] = "adapted" if i in chosen else "base"\n                row["dynamic_multiplier"] = best["multiplier"]\n                row["dynamic_tau"] = tau\n                row["out_ce"] = out_ce[i]\n                f.write(json.dumps(row, ensure_ascii=False) + "\\n")\n\n    summary["dynamic_eval_seconds"] = time.perf_counter() - t0\n    return summary\n\n\ndef main():\n    ap = argparse.ArgumentParser(description="V2-scaled dynamic output-CE routing")\n    ap.add_argument("--dataset", required=True)\n    ap.add_argument("--base-preds", required=True)\n    ap.add_argument("--adapted-preds", required=True)\n    ap.add_argument("--features", required=True)\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--fixed-tau", type=float, default=0.3)\n    ap.add_argument("--multipliers", default="0.25,0.5,0.75,1.0,1.25,1.5,1.75,2.0,2.25,2.5,2.75,3.0,3.25,3.5,3.75,4.0")\n    ap.add_argument("--routed-out", default=None, help="optional JSONL predictions routed at best multiplier")\n    args = ap.parse_args()\n\n    multipliers = [float(x) for x in args.multipliers.split(",") if x.strip()]\n    summary = compute(args.dataset, args.base_preds, args.adapted_preds, args.features, multipliers, args.fixed_tau, args.routed_out)\n    Path(args.out).parent.mkdir(parents=True, exist_ok=True)\n    with open(args.out, "w", encoding="utf-8") as f:\n        json.dump(summary, f, indent=2, ensure_ascii=False)\n\n    print(f"dataset={summary[\'dataset\']} n={summary[\'n\']}")\n    print(f"V2 frac={summary[\'v2_frac\']*100:.1f}%  V2 acc={summary[\'v2_acc\']*100:.1f}%")\n    print(f"best multiplier={summary[\'best_multiplier\']:.2f}x  adapted={summary[\'best_adapted_frac\']*100:.1f}%  "\n          f"tau={summary[\'best_implied_tau\']:.4f}  peak={summary[\'peak_acc\']*100:.1f}%")\n    print(f"fixed CE@{summary[\'fixed_tau\']}={summary[\'fixed_ce_acc\']*100:.1f}%  "\n          f"adapted={summary[\'fixed_ce_adapted_frac\']*100:.1f}%")\n    print(f"dynamic eval={summary[\'dynamic_eval_seconds\']:.3f}s")\n    print(f"-> {args.out}")\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
script_path.chmod(0o755)
print("wrote", script_path)

## Full fresh run with timing

This reproduces the existing CUDA A/B pipeline but saves outputs into `WORK/saves_dynamic_v2ce/`. The slow pieces are pool building, LoRA training, and vLLM generation. The dynamic sweep itself is basically free once `features.jsonl`, `preds_base.jsonl`, and `preds_adapted.jsonl` exist.

Timing behavior:

- Every stage writes elapsed wall-clock seconds into `<dataset>/timing_summary.json`.
- If `SKIP_IF_EXISTS=True`, skipped stages reuse the cached timing from the previous completed run when available.
- The final table includes measured experiment time and an estimated deployment-style routing generation cost.

The deployment estimate is intentionally simple:

```text
estimated routed gen time = base generation time + routed_adapted_fraction × adapted generation time
```

That lets you compare dynamic CE versus fixed `CE@0.3` on cost without forcing the notebook to actually regenerate only a subset for every threshold.

In [ ]:
import subprocess, os, json, time, shlex, sys
from pathlib import Path

SCRIPTS = Path("cuda_ttl/ab_routing").resolve()
OUT_ROOT = Path(WORK) / "saves_dynamic_v2ce"
LOG_ROOT = Path(WORK) / "logs_dynamic_v2ce"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT.mkdir(parents=True, exist_ok=True)

STAGE_ORDER = ["pool", "train", "gen_base", "gen_adapted", "route_eval_fixed", "route_eval_dynamic"]


def _read_json(path, default=None):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return default


def run_cmd(cmd, log_path=None, skip_if=None, cached_stage=None, timing_cache=None):
    """Run a command and return a timing record.

    If the output exists and SKIP_IF_EXISTS is true, this does not rerun the command.
    It reuses cached timing from the previous completed run when available so the final
    table still reflects full-run cost instead of showing skipped stages as free.
    """
    skip_path = Path(skip_if) if skip_if is not None else None
    if skip_path is not None and SKIP_IF_EXISTS and skip_path.exists():
        cached = (timing_cache or {}).get("stages", {}).get(cached_stage or "")
        elapsed = cached.get("elapsed_seconds") if isinstance(cached, dict) else None
        print(f"SKIP exists: {skip_path}" + (f" | cached {elapsed/60:.1f} min" if elapsed else " | no cached timing"))
        return {
            "stage": cached_stage,
            "elapsed_seconds": elapsed,
            "skipped": True,
            "returncode": 0,
            "cmd": [str(c) for c in cmd],
        }

    print("RUN:", " ".join(shlex.quote(str(c)) for c in cmd))
    t0 = time.perf_counter()
    if log_path:
        Path(log_path).parent.mkdir(parents=True, exist_ok=True)
        with open(log_path, "w", encoding="utf-8") as log:
            p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            for line in p.stdout:
                print(line, end="")
                log.write(line)
            rc = p.wait()
    else:
        rc = subprocess.call(cmd)
    elapsed = time.perf_counter() - t0
    print(f"rc={rc} elapsed={elapsed/60:.1f} min")
    if rc != 0:
        raise RuntimeError(f"Command failed with rc={rc}: {cmd}")
    return {
        "stage": cached_stage,
        "elapsed_seconds": elapsed,
        "skipped": False,
        "returncode": rc,
        "cmd": [str(c) for c in cmd],
    }


def _sum_known_seconds(stage_records, names):
    vals = [stage_records.get(n, {}).get("elapsed_seconds") for n in names]
    if any(v is None for v in vals):
        return None
    return float(sum(vals))


def write_timing_summary(ds, d, stage_records):
    timing_path = d / "timing_summary.json"
    full_pipeline = _sum_known_seconds(stage_records, STAGE_ORDER)
    measured_this_notebook = sum(
        r.get("elapsed_seconds") or 0.0
        for r in stage_records.values()
        if not r.get("skipped")
    )
    summary = {
        "dataset": ds,
        "timing_note": "elapsed_seconds are wall-clock seconds; skipped stages reuse cached elapsed_seconds when available",
        "full_pipeline_seconds": full_pipeline,
        "measured_this_notebook_seconds": measured_this_notebook,
        "stages": stage_records,
    }
    with open(timing_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    print("wrote", timing_path)
    return timing_path


def run_full_dataset(ds):
    n = N_BY_DATASET[ds]
    data = DATA_FILES[ds]
    d = OUT_ROOT / ds
    pool = d / "pool"
    adapter = d / "adapter"
    base_preds = d / "preds_base.jsonl"
    adapted_preds = d / "preds_adapted.jsonl"
    eval_json = d / "routing_eval.json"
    dyn_json = d / "dynamic_ce_from_v2.json"
    routed_jsonl = d / "preds_dynamic_best.jsonl"
    timing_path = d / "timing_summary.json"
    d.mkdir(parents=True, exist_ok=True)

    old_timing = _read_json(timing_path, default={}) or {}
    stage_records = dict(old_timing.get("stages", {}))

    def record(stage, cmd, log_path, skip_if):
        rec = run_cmd(cmd, log_path, skip_if=skip_if, cached_stage=stage, timing_cache=old_timing)
        stage_records[stage] = rec
        write_timing_summary(ds, d, stage_records)
        return rec

    record("pool",
           [sys.executable, str(SCRIPTS / "ab_build_pool.py"),
            "--model", MODEL, "--data", data, "--dataset", ds,
            "--max-samples", str(n), "--out-dir", str(pool)],
           LOG_ROOT / f"{ds}_pool.log", pool / "features.jsonl")

    record("train",
           [sys.executable, str(SCRIPTS / "ab_train_ttl.py"),
            "--model", MODEL, "--data", data, "--selection-file", str(pool / "selection.json"),
            "--max-samples", str(n), "--output-dir", str(adapter)],
           LOG_ROOT / f"{ds}_train.log", adapter / "merged" / "config.json")

    record("gen_base",
           [sys.executable, str(SCRIPTS / "ab_generate.py"),
            "--model", MODEL, "--data", data, "--max-samples", str(n), "--output", str(base_preds)],
           LOG_ROOT / f"{ds}_gen_base.log", base_preds)

    record("gen_adapted",
           [sys.executable, str(SCRIPTS / "ab_generate.py"),
            "--model", str(adapter / "merged"), "--data", data, "--max-samples", str(n), "--output", str(adapted_preds)],
           LOG_ROOT / f"{ds}_gen_adapted.log", adapted_preds)

    record("route_eval_fixed",
           [sys.executable, str(SCRIPTS / "ab_route_eval.py"),
            "--dataset", ds, "--base-preds", str(base_preds), "--adapted-preds", str(adapted_preds),
            "--features", str(pool / "features.jsonl"), "--out", str(eval_json)],
           LOG_ROOT / f"{ds}_route_eval.log", eval_json)

    # Do not skip this stage by default if you changed multipliers/FIXED_CE_TAU and want a refresh.
    # With SKIP_IF_EXISTS=True, delete dynamic_ce_from_v2.json to force recalculation.
    record("route_eval_dynamic",
           [sys.executable, str(SCRIPTS / "ab_dynamic_ce_from_v2.py"),
            "--dataset", ds, "--base-preds", str(base_preds), "--adapted-preds", str(adapted_preds),
            "--features", str(pool / "features.jsonl"), "--fixed-tau", str(FIXED_CE_TAU),
            "--multipliers", ",".join(map(str, MULTIPLIERS)),
            "--out", str(dyn_json), "--routed-out", str(routed_jsonl)],
           LOG_ROOT / f"{ds}_dynamic_ce.log", dyn_json)

    final_timing_path = write_timing_summary(ds, d, stage_records)

    # Attach timing to the dynamic summary so the final table is one-file friendly.
    if dyn_json.exists():
        dyn = _read_json(dyn_json, default={}) or {}
        dyn["timing_summary_path"] = str(final_timing_path)
        dyn["timing"] = _read_json(final_timing_path, default={}) or {}
        with open(dyn_json, "w", encoding="utf-8") as f:
            json.dump(dyn, f, indent=2, ensure_ascii=False)

    return dyn_json


if RUN_FULL_PIPELINE:
    dynamic_jsons = [run_full_dataset(ds) for ds in DATASETS]
else:
    dynamic_jsons = []
    print("RUN_FULL_PIPELINE=False, skipping train/generate pipeline")

## Fast replay fallback

If you are just checking the logic and the repo already contains `results/ab_routing_2026-07-02/*_routing_eval.json`, this cell converts those slim per-item evals into the same dynamic table without retraining. This is only a smoke test; the previous section is the true fresh run.

In [ ]:
def dynamic_from_existing_routing_eval(path, fixed_tau=0.3, multipliers=MULTIPLIERS):
    d = json.load(open(path, encoding="utf-8"))
    items = d["items"]
    idx = [int(x["idx"]) for x in items]
    base_ok = {int(x["idx"]): bool(x["base_ok"]) for x in items}
    adapted_ok = {int(x["idx"]): bool(x["adapted_ok"]) for x in items}
    out_ce = {int(x["idx"]): float(x["out_ce"]) for x in items}
    v2_set = {int(x["idx"]) for x in items if x.get("route_choice") == "adapted"}
    n = len(idx)
    v2_frac = len(v2_set) / n
    order = sorted(idx, key=lambda i: out_ce[i], reverse=True)

    def acc(sel):
        return sum((adapted_ok[i] if i in sel else base_ok[i]) for i in idx) / n

    rows = []
    for m in multipliers:
        frac = min(1.0, v2_frac * float(m))
        k = int(round(n * frac))
        chosen = set(order[:k])
        tau = float("inf") if k == 0 else out_ce[order[min(k, n) - 1]]
        rows.append({
            "dataset": d["dataset"],
            "multiplier": float(m),
            "v2_frac": v2_frac,
            "adapted_frac": k / n,
            "implied_tau": tau,
            "acc": acc(chosen),
        })
    best = sorted(rows, key=lambda r: (-r["acc"], r["adapted_frac"], r["multiplier"]))[0]
    fixed_set = {i for i in idx if out_ce[i] >= fixed_tau}
    return {
        "dataset": d["dataset"],
        "n": n,
        "v2_frac": v2_frac,
        "best_multiplier": best["multiplier"],
        "best_adapted_frac": best["adapted_frac"],
        "best_implied_tau": best["implied_tau"],
        "peak_acc": best["acc"],
        "fixed_ce_acc": acc(fixed_set),
        "fixed_ce_adapted_frac": len(fixed_set) / n,
        "sweep": rows,
    }

if not dynamic_jsons:
    existing = {
        "logiqa": Path("results/ab_routing_2026-07-02/logiqa_routing_eval.json"),
        "medicine": Path("results/ab_routing_2026-07-02/medicine_routing_eval.json"),
        "geography": Path("results/ab_routing_2026-07-02/geography_routing_eval.json"),
    }
    replay_summaries = []
    for ds in DATASETS:
        if existing[ds].exists():
            replay_summaries.append(dynamic_from_existing_routing_eval(existing[ds], FIXED_CE_TAU))
    print("replayed", len(replay_summaries), "existing result files")
else:
    replay_summaries = []

## Final table with accuracy + compute

The final table has two kinds of timing:

- **Measured full pipeline min**: pool + train + base generation + adapted generation + fixed eval + dynamic eval. This is the cost to run the whole experiment notebook for that dataset.
- **Estimated routed gen min**: deployment-style comparison using measured generation time and routed fraction. This is the useful “which route is cheaper?” column.

Because the notebook has to generate both base and adapted predictions for offline comparison, the routed-generation number is an estimate, not a separately executed subset-generation run.

In [ ]:
import pandas as pd
import math

if dynamic_jsons:
    summaries = [json.load(open(p, encoding="utf-8")) for p in dynamic_jsons]
elif replay_summaries:
    summaries = replay_summaries
else:
    raise RuntimeError("No dynamic summaries found. Run the full pipeline or use a repo snapshot with existing results.")


def safe_get(dct, path, default=None):
    cur = dct
    for key in path:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur


def sec_to_min(x):
    return None if x is None else x / 60.0

rows = []
for s in summaries:
    ds = s["dataset"]
    timing = s.get("timing") or {}
    stages = timing.get("stages", {})
    pool_sec = safe_get(stages, ["pool", "elapsed_seconds"])
    train_sec = safe_get(stages, ["train", "elapsed_seconds"])
    gen_base_sec = safe_get(stages, ["gen_base", "elapsed_seconds"])
    gen_adapted_sec = safe_get(stages, ["gen_adapted", "elapsed_seconds"])
    fixed_eval_sec = safe_get(stages, ["route_eval_fixed", "elapsed_seconds"])
    dyn_eval_sec = safe_get(stages, ["route_eval_dynamic", "elapsed_seconds"], s.get("dynamic_eval_seconds"))
    full_pipeline_sec = timing.get("full_pipeline_seconds")
    measured_this_nb_sec = timing.get("measured_this_notebook_seconds")

    best_frac = s.get("best_adapted_frac", s.get("adapted_frac"))
    fixed_frac = s.get("fixed_ce_adapted_frac")

    # Deployment-style routed generation estimates. Assumes base generation/answering is required for all rows,
    # and the adapted model is only run for the routed fraction.
    dyn_routed_gen_sec = None
    fixed_routed_gen_sec = None
    dyn_vs_fixed_savings = None
    if gen_base_sec is not None and gen_adapted_sec is not None and best_frac is not None:
        dyn_routed_gen_sec = gen_base_sec + best_frac * gen_adapted_sec
    if gen_base_sec is not None and gen_adapted_sec is not None and fixed_frac is not None:
        fixed_routed_gen_sec = gen_base_sec + fixed_frac * gen_adapted_sec
    if dyn_routed_gen_sec is not None and fixed_routed_gen_sec:
        dyn_vs_fixed_savings = 1.0 - (dyn_routed_gen_sec / fixed_routed_gen_sec)

    rows.append({
        "dataset": ds,
        "n": s.get("n"),
        "V2 frac": s["v2_frac"],
        "best multiplier": s["best_multiplier"],
        "adapted frac": best_frac,
        "implied tau": s["best_implied_tau"],
        "peak acc": s["peak_acc"],
        "fixed CE@0.3": s["fixed_ce_acc"],
        "fixed CE adapted frac": fixed_frac,
        "pool min": sec_to_min(pool_sec),
        "train min": sec_to_min(train_sec),
        "base gen min": sec_to_min(gen_base_sec),
        "adapted gen min": sec_to_min(gen_adapted_sec),
        "fixed eval sec": fixed_eval_sec,
        "dynamic eval sec": dyn_eval_sec,
        "measured full pipeline min": sec_to_min(full_pipeline_sec),
        "measured this notebook min": sec_to_min(measured_this_nb_sec),
        "est dynamic routed gen min": sec_to_min(dyn_routed_gen_sec),
        "est fixed routed gen min": sec_to_min(fixed_routed_gen_sec),
        "dynamic savings vs fixed": dyn_vs_fixed_savings,
    })

df = pd.DataFrame(rows).sort_values("dataset")

# Pretty display like the target table, now with cost columns.
display_df = df.copy()
for col in ["V2 frac", "adapted frac", "peak acc", "fixed CE@0.3", "fixed CE adapted frac", "dynamic savings vs fixed"]:
    display_df[col] = display_df[col].map(lambda x: f"{100*x:.1f}%" if pd.notnull(x) else "")
display_df["best multiplier"] = display_df["best multiplier"].map(lambda x: f"{x:.2f}x")
display_df["implied tau"] = display_df["implied tau"].map(lambda x: f"{x:.4f}")
for col in ["pool min", "train min", "base gen min", "adapted gen min", "measured full pipeline min", "measured this notebook min", "est dynamic routed gen min", "est fixed routed gen min"]:
    display_df[col] = display_df[col].map(lambda x: f"{x:.2f}" if pd.notnull(x) else "")
for col in ["fixed eval sec", "dynamic eval sec"]:
    display_df[col] = display_df[col].map(lambda x: f"{x:.2f}" if pd.notnull(x) else "")

main_cols = [
    "dataset", "V2 frac", "best multiplier", "adapted frac", "implied tau",
    "peak acc", "fixed CE@0.3", "est dynamic routed gen min", "est fixed routed gen min",
    "dynamic savings vs fixed", "measured full pipeline min"
]
display(display_df[main_cols])

# Optional expanded timing table.
timing_cols = [
    "dataset", "pool min", "train min", "base gen min", "adapted gen min",
    "fixed eval sec", "dynamic eval sec", "measured full pipeline min", "measured this notebook min"
]
display(display_df[timing_cols])

out_csv = OUT_ROOT / "dynamic_v2_ce_summary_with_compute.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_csv, index=False)
print("wrote", out_csv)

In [ ]:
# Per-dataset multiplier curves
import matplotlib.pyplot as plt

for s in summaries:
    sweep = s["sweep"]
    xs = [r["multiplier"] for r in sweep]
    ys = [100 * r["acc"] for r in sweep]
    plt.figure()
    plt.plot(xs, ys, marker="o")
    plt.xlabel("V2 fraction multiplier")
    plt.ylabel("routed accuracy (%)")
    plt.title(f"{s['dataset']}: dynamic CE routing multiplier sweep")
    plt.grid(True)
    plt.show()

## What to report

Use the final table for the main comparison. The interpretation should be:

- V2 admission routing is a useful **fraction anchor**, but too conservative as an inference route.
- CE should remain the routing signal. The multiplier simply turns V2's admission fraction into a data-dependent CE percentile.
- Compute-wise, compare `est dynamic routed gen min` against `est fixed routed gen min` when asking which route is cheapest at inference time.
- `measured full pipeline min` is the experimental cost to produce the evidence, not the expected deployed routing cost.
- The best multiplier in this notebook is test-tuned. To avoid leakage, split the data, pick `multiplier` on validation, freeze it, and report test.
- If dynamic CE beats or matches fixed CE but routes a larger adapted fraction, it may be accuracy-neutral but not cheaper. If it matches accuracy with a smaller adapted fraction, that is the clean win.
